## Imports and loading

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MinMaxScaler

from sklearn.impute import SimpleImputer
import html
import re

print("Caricamento dati...")


train_df = pd.read_csv("development.csv", na_values=['\\N', '0000-00-00 00:00:00', '  '])
eval_df = pd.read_csv("evaluation.csv", na_values=['\\N', '0000-00-00 00:00:00', '  '])

print(train_df.isna().sum())



Caricamento dati...
Id               0
source         294
title            3
article       1878
page_rank        0
timestamp    27750
label            0
dtype: int64


In [ ]:
import re
import html

from html.parser import HTMLParser

def clean_text(text):
    if not isinstance(text, str):
        return str(text)

    
    text = html.unescape(text)

    return text

train_df['title'] = train_df['title'].fillna('')
train_df['article'] = train_df['article'].fillna('')

eval_df['title'] = eval_df['title'].fillna('')
eval_df['article'] = eval_df['article'].fillna('')

train_df['title'] = train_df['title'].apply(clean_text)
eval_df['title'] = eval_df['title'].apply(clean_text)

train_df['article'] = train_df['article'].apply(clean_text)
eval_df['article'] = eval_df['article'].apply(clean_text)


train_df['text_content'] = train_df['title']+ ' ' + train_df['article']

eval_df['text_content'] = eval_df['title']+ ' ' + eval_df['article']

print("Dati caricati.")


Dati caricati.


In [3]:
print(f"Dataset originale: {len(train_df)} righe.")
train_df = train_df[train_df['text_content'] != " "]
print(f"Dataset dopo rimozione righe con title e article vuoti: {len(train_df)} righe.")


group_cols = ['text_content']

# 1. Funzione per determinare la moda o segnare il pareggio
# Se len(x.mode()) == 1, c'è una maggioranza chiara -> restituisce la moda.
# Altrimenti (len > 1), c'è un pareggio -> restituisce -1.
def get_majority_or_tie(x):
    modes = x.mode()
    if len(modes) == 1:
        return modes[0]
    else:
        return -1


# 2. Applica la logica
voted_labels = train_df.groupby(group_cols)['label'].transform(get_majority_or_tie)

# 3. Sovrascrivi la colonna label
train_df['label'] = voted_labels

# 4. Elimina PRIMA i pareggi (tutte le righe con -1 vengono scartate)
# Assicurati che -1 non sia una label valida nel tuo dataset originale!
train_df = train_df[train_df['label'] != -1]

# 5. Elimina i duplicati
# Ora rimangono solo gli articoli con una maggioranza chiara.
# Possiamo tenere la prima riga di ogni gruppo.
train_df = train_df.drop_duplicates(subset=group_cols, keep='first')

print(f"Dataset pulito: {len(train_df)} righe (pareggi eliminati).")




group_cols = ['article']

# 1. CREA UNA MASCHERA per separare i dati
# True per le righe da elaborare, False per quelle da ignorare (title vuoto)
mask_process = train_df['article'].str.strip() != ''

# 2. SEPARA IL DATAFRAME IN DUE PEZZI
df_to_process = train_df[mask_process].copy() # Righe con titolo valido
df_ignored = train_df[~mask_process].copy()   # Righe con title='' (da conservare intatte)

# 3. APPLICA LA LOGICA SOLO SU 'df_to_process'
if not df_to_process.empty:
    # Calcola la maggioranza
    voted_labels = df_to_process.groupby(group_cols)['label'].transform(get_majority_or_tie)
    
    # Assegna le nuove label
    df_to_process['label'] = voted_labels
    
    # Elimina i pareggi (-1)
    df_to_process = df_to_process[df_to_process['label'] != -1]
    
    # Elimina i duplicati (mantenendo una sola riga per titolo)
    df_to_process = df_to_process.drop_duplicates(subset=group_cols, keep='first')

# 4. RIUNISCI I DUE PEZZI
# Concateniamo le righe elaborate (pulite e deduplicate) con quelle ignorate (originali)
train_df = pd.concat([df_to_process, df_ignored], ignore_index=True)

print(f"Dataset finale dopo majority voting su 'article': {len(train_df)} righe.")

group_cols = ['title']

# 1. CREA UNA MASCHERA per separare i dati
# True per le righe da elaborare, False per quelle da ignorare (title vuoto)
mask_process = train_df['title'].str.strip() != ' '

# 2. SEPARA IL DATAFRAME IN DUE PEZZI
df_to_process = train_df[mask_process].copy() # Righe con titolo valido
df_ignored = train_df[~mask_process].copy()   # Righe con title='' (da conservare intatte)

# 3. APPLICA LA LOGICA SOLO SU 'df_to_process'
if not df_to_process.empty:
    # Calcola la maggioranza
    voted_labels = df_to_process.groupby(group_cols)['label'].transform(get_majority_or_tie)
    
    # Assegna le nuove label
    df_to_process['label'] = voted_labels
    
    # Elimina i pareggi (-1)
    df_to_process = df_to_process[df_to_process['label'] != -1]
    
    # Elimina i duplicati (mantenendo una sola riga per titolo)
    df_to_process = df_to_process.drop_duplicates(subset=group_cols, keep='first')

# 4. RIUNISCI I DUE PEZZI
# Concateniamo le righe elaborate (pulite e deduplicate) con quelle ignorate (originali)
train_df = pd.concat([df_to_process, df_ignored], ignore_index=True)

print(f"Dataset finale dopo majority voting su 'title': {len(train_df)} righe.")

#

Dataset originale: 79997 righe.
Dataset dopo rimozione righe con title e article vuoti: 79995 righe.
Dataset pulito: 75347 righe (pareggi eliminati).
Dataset finale dopo majority voting su 'article': 74149 righe.
Dataset finale dopo majority voting su 'title': 72857 righe.


## Preprocessing

In [4]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder


text_transformer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=3,
    max_df=0.5
    )

text_transformer2 = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    )

cat_transformer = OneHotEncoder(handle_unknown='ignore')

hour_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ])
source_pipe = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(max_categories=51, handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('txt', text_transformer, 'text_content'),
        #('txt2', text_transformer2, 'title'),
        ('ord', cat_transformer, ['page_rank']),
        ('hour', hour_transformer, ['hour_cos','hour_sin']),
        ('cat_source', source_pipe, ['source']),
        
    ]
)
train_df['day_of_week'] = pd.to_datetime(train_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.dayofweek
eval_df['day_of_week'] = pd.to_datetime(eval_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.dayofweek

train_df['is_weekend'] = train_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
train_df['timestamp_missing'] = train_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)
eval_df['is_weekend'] = eval_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
eval_df['timestamp_missing'] = eval_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)


train_df['hour_of_day'] = pd.to_datetime(train_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.hour
eval_df['hour_of_day'] = pd.to_datetime(eval_df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S').dt.hour
train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour_of_day'] / 24)
train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour_of_day'] / 24)

eval_df['hour_sin'] = np.sin(2 * np.pi * eval_df['hour_of_day'] / 24)
eval_df['hour_cos'] = np.cos(2 * np.pi * eval_df['hour_of_day'] / 24)

X_eval = eval_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]


X = train_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]
y = train_df['label']

print(train_df['source'].isna().sum())


272


## Linear SVC

In [5]:

 ## LINEAR SVC MODEL

classifier = LinearSVC(random_state=42)
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])


param_grid = {
    'clf__C': [0.1],
    'clf__class_weight': ['balanced'],
    'preprocessor__txt__ngram_range': [(1,3),(1,4)],
    'preprocessor__txt__min_df':[2,3,4,5],
    'preprocessor__txt__max_df': [0.4, 0.5, 0.6, 0.7],
    'preprocessor__cat_source__encoder__max_categories': [300, 400, 500] # ora è fissato a 300, poi mettiamo più valori

}

grid_search = GridSearchCV(
    model_pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2
)

grid_search.fit(X, y)
print("LINEAR SVC - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_



Fitting 5 folds for each of 96 candidates, totalling 480 fits
LINEAR SVC - Grid Search Results:
{'clf__C': 0.1, 'clf__class_weight': 'balanced', 'preprocessor__cat_source__encoder__max_categories': 300, 'preprocessor__txt__max_df': 0.4, 'preprocessor__txt__min_df': 3, 'preprocessor__txt__ngram_range': (1, 3)}
Best Cross-Validation Macro F1: 0.7041


In [6]:
best_model.fit(X, y)
y_eval_pred = best_model.predict(X_eval)

pd.DataFrame({
    'Id': eval_df['Id'],
    'Predicted': y_eval_pred}).to_csv('submission.csv', index=False)

# Random Forest


In [7]:

from sklearn.ensemble import RandomForestClassifier


classifier = RandomForestClassifier(random_state=42)
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])


param_grid = {
    'clf__max_features': ['sqrt', 'log2'],
    'clf__min_samples_split': [5, 10, 20],
    'clf__max_depth': [None, 10, 20, 30],
    'clf__n_estimators': [100, 200],
    'clf__class_weight': ['balanced'],
    'preprocessor__txt__ngram_range': [(1,3)],
    'preprocessor__cat_source__encoder__max_categories': [301] # ora è fissato a 300, poi mettiamo più valori

}

grid_search = GridSearchCV(
    model_pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2
)

grid_search.fit(X, y)
print("LINEAR SVC - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_

Fitting 5 folds for each of 48 candidates, totalling 240 fits


KeyboardInterrupt: 

# HistGradientBoosting


In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import KNNImputer
from sklearn.pipeline import FunctionTransformer
from sklearn.preprocessing import OrdinalEncoder

text_transformer = Pipeline([
    ('tfidf', TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    max_features=2000
    ))
    ])





cat_transformer = OneHotEncoder(handle_unknown='ignore')

hour_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ])
source_pipe = Pipeline([
    ('encoder', OneHotEncoder(max_categories=301, handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('txt', text_transformer, 'text_content'),
        ('ord', cat_transformer, ['page_rank']),
        ('hour', hour_transformer, ['hour_cos','hour_sin']),
        ('cat_source', source_pipe, ['source']),
    ],
    sparse_threshold=0
)
train_df['day_of_week'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.dayofweek
eval_df['day_of_week'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.dayofweek

train_df['is_weekend'] = train_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
train_df['timestamp_missing'] = train_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)
eval_df['is_weekend'] = eval_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
eval_df['timestamp_missing'] = eval_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)

train_df['text_lenght'] = train_df['text_content'].apply(lambda x: len(x))
eval_df['text_lenght'] = eval_df['text_content'].apply(lambda x: len(x))
train_df['hour_of_day'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.hour
eval_df['hour_of_day'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.hour
train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour_of_day'] / 24)
train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour_of_day'] / 24)

eval_df['hour_sin'] = np.sin(2 * np.pi * eval_df['hour_of_day'] / 24)
eval_df['hour_cos'] = np.cos(2 * np.pi * eval_df['hour_of_day'] / 24)

X_eval = eval_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]


X = train_df[['text_content','source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]
y = train_df['label']


In [ ]:


from sklearn.ensemble import HistGradientBoostingClassifier

classifier = HistGradientBoostingClassifier(random_state=42, early_stopping=True)
model_pipeline = Pipeline([
    ('clf', classifier)
])
X=preprocessor.fit_transform(X)
X_eval=preprocessor.transform(X_eval)


param_grid = {
    'learning_rate': [0.1],       # Quanto velocemente impara
    'max_iter': [100],             # Numero massimo di alberi (simile a n_estimators)
    'max_depth': [5, 10],               # Profondità massima (None = illimitata)
    'min_samples_leaf': [20],            # Minimo numero di campioni per foglia (regolarizzazione)
    'l2_regularization': [0, 1.0],      # Regolarizzazione L2 (Ridge)
    'class_weight': ['balanced'],
    #'preprocessor__txt__tfidf__ngram_range': [(1,3)],
    #'preprocessor__cat_source__imputer_top_n__top_n': [50, 300] # ora è fissato a 300, poi mettiamo più valori

}

grid_search = GridSearchCV(
    classifier, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=1, 
    verbose=2
)

grid_search.fit(X, y)
print("LINEAR SVC - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_



Fitting 5 folds for each of 4 candidates, totalling 20 fits


KeyboardInterrupt: 